# Feature selection EDA

Lets look at the 8 features kept by `src/gbm_feature_selection.py`, then at `occu_h` for question 6

In [1]:
from lib import eda
import json
import pandas as pd

# Import the data
df = pd.read_pickle("../data/AML_data_transformed.pkl")

# Features kept by the selection
with open("../results/gbm_feature_selection/selected_features.json") as f:
    kept = json.load(f)["features"]

kept

['in_person_visit_cnt',
 'tot_cross_border_val_12m',
 'txn_cash_out_cnt_3m',
 'avg_cash_5m',
 'geo_mail_addr_country_rate_high',
 'prod_cda',
 'prod_total_acct_cnt',
 'txn_wires_in_high_cnt_12m']

## in_person_visit_cnt
Number of in-person interaction with the customer



In [2]:
eda(df, "rating", "in_person_visit_cnt", bins = 3)

## avg_cash_5m
Average of cash transaction (withdrawal and deposit) in the past 5 months

In [3]:
eda(df, "rating", "avg_cash_5m", bins = 3)

## geo_mail_addr_country_rate_high
Binary indicator for mailing address being high risk country

In [4]:
eda(df, "rating", "geo_mail_addr_country_rate_high")

## tot_cross_border_val_12m
Total amount of cross-border wire transaction (in and out) for the past 12 months

In [5]:
eda(df, "rating", "tot_cross_border_val_12m", bins = 2)

## prod_total_acct_cnt
Number of accounts

In [6]:
eda(df, "rating", "prod_total_acct_cnt", bins = 2)

## prod_cda
Binary indicator for chequing account

In [7]:
eda(df, "rating", "prod_cda")

## txn_cash_out_cnt_3m
Total number of cash transactions withdrawn in the last 3 months

In [8]:
eda(df, "rating", "txn_cash_out_cnt_3m", bins = 3)

Weird one: the share of high risk goes down a bit with more withdrawals (34% to 29%). We would expect the opposite, see question 6 of the report

## txn_wires_in_high_cnt_12m
Count of wires received from high risk countries in the last 12 months

In [9]:
eda(df, "rating", "txn_wires_in_high_cnt_12m", bins = 2)

## occu_h (not kept)
Binary indicator of high risk occupation. It's the example of question 6: a high risk occupation should mean a higher rating

In [10]:
eda(df, "rating", "occu_h")

In [11]:
# Share of high risk with and without a high risk occupation, and a test to see if the gap is real
from scipy.stats import fisher_exact

print(pd.crosstab(df["occu_h"], df["rating"], normalize = "index").round(3))
print("Fisher p-value:", round(fisher_exact(pd.crosstab(df["occu_h"], df["rating"]).values)[1], 3))

rating      0      1
occu_h              
0       0.646  0.354
1       0.682  0.318
Fisher p-value: 0.383


It's the opposite: 32% of high risk with a high risk occupation vs 35% without. The gap is small though (p = 0.38), so the flag just doesn't say much here, and 48% of clients have it, way too broad. The selection didn't keep it

## Plots for the report

In [ ]:
# Save the plots used in the report
import os
os.makedirs("../results/eda", exist_ok = True)

report_plots = {"in_person_visit_cnt": 3, "avg_cash_5m": 3, "tot_cross_border_val_12m": 2,
                "geo_mail_addr_country_rate_high": 2, "txn_cash_out_cnt_3m": 3}

for feature, bins in report_plots.items():
    eda(df, "rating", feature, bins = bins).write_image(f"../results/eda/{feature}.png", scale = 2)